In [1]:
import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM
from filterpy.kalman import KalmanFilter
import matplotlib.pyplot as plt
import logging


logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class RegimeSwitchModel:
    def __init__(self, symbol, interval='1d', train_pct=0.8, strategy='long-only'):
        self.symbol = symbol
        self.interval = interval
        self.train_pct = train_pct
        if strategy not in ['long-only', 'long-short']:
            raise ValueError("Strategy must be either 'long-only' or 'long-short'")
        self.strategy = strategy

    def compute_features(self, data, features_config):
        df = data.copy()
        for feature, config in features_config.items():
            if feature == 'log_return':
                df['log_return'] = np.log(df['Close']).pct_change()
            elif feature == 'lnrange':
                df['lnrange'] = np.log(df['High'] / df['Low'])
            elif feature == 'rsi':
                period = config.get('period', 14)
                delta = df['Close'].diff()
                gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
                loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
                rs = gain / loss
                df['rsi'] = 100 - (100 / (1 + rs))
                
        return df.dropna()
    
    def load_macro_data(self, macro_features):
        all_macro_data = []

        for feature in macro_features:
            print(f"Loading {feature} data...")
            macro_df = pd.read_csv(f"/Users/valter.rebelo/MissionControl/data/macro/fredData/{feature}.csv")
            macro_df['date'] = pd.to_datetime(macro_df['date'])
            
            # Check frequency by calculating median time delta
            time_deltas = macro_df['date'].diff().dropna()
            median_delta = time_deltas.median().days
            
            if median_delta > 1:
                logging.info(f"Feature {feature} has {median_delta}-day frequency. Last date: {macro_df['date'].max()}")
                
                # Resample to daily frequency
                macro_df.set_index('date', inplace=True)
                macro_df = macro_df.resample('D').ffill().bfill()
                macro_df.reset_index(inplace=True)
            
            all_macro_data.append(macro_df.set_index('date'))
            
        # Combine all macro features
        combined_macro_df = pd.concat(all_macro_data, axis=1)
        
        return combined_macro_df

    def load_data(self):
        try:
            df_1 = pd.read_csv(f"/Users/valter.rebelo/MissionControl/data/micro/candleData/{self.symbol}_candles.csv")
            df_1['date'] = pd.to_datetime(df_1['date'])
            df_1.set_index('date', inplace=True)

            df_2 = pd.read_csv(f"/Users/valter.rebelo/MissionControl/data/micro/assetData/{self.symbol}.csv")
            df_2['date'] = pd.to_datetime(df_2['date'])
            df_2.set_index('date', inplace=True)

            btc_df = pd.read_csv("/Users/valter.rebelo/MissionControl/data/micro/candleData/bitcoin_candles.csv")
            btc_df['date'] = pd.to_datetime(btc_df['date'])
            btc_df.set_index('date', inplace=True)

            data = pd.merge(df_1, df_2[['total_volume', 'market_cap']], on='date', how='inner')
            data.rename(columns={'total_volume': 'Volume', 'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close'}, inplace=True)
            data.index.name = 'Date'

            if self.symbol != "bitcoin":
                data['close_btc'] = (data['Close'] / btc_df['close']) * 100
                data.dropna(inplace=True)

            if self.symbol == "bitcoin":
                data = data[data.index >= '2017-01-01']

            # Compute log_close for Kalman Filter
            data['log_close'] = np.log(data['Close'])

            # Compute dynamic features
            features_config = {
                'log_return': {},
                'lnrange': {},
                'rsi': {'period': 14}
            }

            macro_features = ['move', 'treasury5YInflationExpectation', 'creditSpreads', 'vix']
            
            macro_df = self.load_macro_data(macro_features)

            # Merge macro data using index
            data = pd.merge(data, macro_df, 
                        left_index=True, right_index=True, 
                        how='left')

            data = self.compute_features(data, features_config)

            if len(data) < 100:
                raise ValueError("Insufficient data points (<100) for meaningful analysis.")
            logging.info(f"Loaded {len(data)} rows for {self.symbol}")
            return data
        except Exception as e:
            logging.error(f"Data loading failed for {self.symbol}: {str(e)}")
            raise

    def split_data(self, data, embargo_percent=0.01):
        try:
            total_rows = len(data)
            train_end_idx = int(total_rows * self.train_pct)
            embargo_end_idx = int(train_end_idx + (total_rows * embargo_percent))
            
            train = data.iloc[:train_end_idx]
            embargo = data.iloc[train_end_idx:embargo_end_idx]
            test = data.iloc[embargo_end_idx:]
            
            self.train_data = train

            if len(train) < 50 or len(test) < 50:
                raise ValueError("Train or test set too small (<50 rows).")
            logging.info(f"Split data: train={len(train)}, embargo={len(embargo)}, test={len(test)}")
            return train, embargo, test
        except Exception as e:
            logging.error(f"Data splitting failed: {str(e)}")
            raise

    def normalize_features(self, train, test, features):

        train_mean = train[features].mean()
        train_std = train[features].std()
        train_normalized = (train[features] - train_mean) / train_std
        test_normalized = (test[features] - train_mean) / train_std

        return (pd.DataFrame(train_normalized, index=train.index, columns=features),
                pd.DataFrame(test_normalized, index=test.index, columns=features))

    def train_hmm(self, train, features):
        try:
            f_train_normalized, _ = self.normalize_features(train, train, features)
            hmm = GaussianHMM(n_components=3, covariance_type='diag', n_iter=500, random_state=42)
            hmm.fit(f_train_normalized)
            if not hmm.monitor_.converged:
                logging.warning("HMM training did not converge.")
            logging.info("HMM trained successfully")
            return hmm
        except Exception as e:
            logging.error(f"HMM training failed: {str(e)}")
            raise

    def train_kf(self, test):
        try:
            kf = KalmanFilter(dim_x=2, dim_z=1)
            kf.F = np.array([[1, 1], [0, 1]])
            kf.H = np.array([[1, 0]])
            kf.Q = np.eye(2) * 0.01
            kf.R = np.array([[10]])
            kf.x = np.array([test['log_close'].iloc[0], 0])
            kf.P = np.eye(2) * 1000
            mu, _, _, _ = kf.batch_filter(test['log_close'].values)
            kf_slope_raw = np.diff(mu[:, 0])
            kf_slope = np.concatenate([[0], kf_slope_raw])
            logging.info("Kalman Filter trained successfully")
            return mu[:, 0], kf_slope
        except Exception as e:
            logging.error(f"Kalman Filter training failed: {str(e)}")
            raise

    def resample_weekly(self, data):
        weekly = data.resample('W-MON').agg({
            'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 
            'Volume': 'sum', 'log_close': 'last', 'log_return': 'sum', 
            'lnrange': 'mean', 'rsi': 'mean', 'creditSpreads': 'last', 
            'move': 'last', 'treasury5YInflationExpectation': 'last', 'vix': 'last'
        })
        return weekly.dropna()

    def train_models_multi_res(self, train, test, features):
        try:
            f_train_daily, f_test_daily = self.normalize_features(train, test, features)
            hmm_daily = GaussianHMM(n_components=3, covariance_type='full', n_iter=50000, random_state=42)
            hmm_daily.fit(f_train_daily)
            kf_est_daily, kf_slope_daily = self.train_kf(test)

            train_weekly = self.resample_weekly(train)
            test_weekly = self.resample_weekly(test)
            f_train_weekly, f_test_weekly = self.normalize_features(train_weekly, test_weekly, features)
            hmm_weekly = GaussianHMM(n_components=3, covariance_type='full', n_iter=50000, random_state=42)
            hmm_weekly.fit(f_train_weekly)
            kf_est_weekly, kf_slope_weekly = self.train_kf(test_weekly)

            logging.info("Multi-resolution models trained successfully")
            return (hmm_daily, f_test_daily, kf_slope_daily), (hmm_weekly, f_test_weekly, kf_slope_weekly)
        except Exception as e:
            logging.error(f"Multi-resolution training failed: {str(e)}")
            raise

    def voting_machine(self, hmm_daily, f_test_daily, kf_slope_daily, 
                      hmm_weekly, f_test_weekly, kf_slope_weekly, test_daily):
        
        # Daily predictions
        hidden_states_daily = hmm_daily.predict(f_test_daily)
        state_means_daily = hmm_daily.means_[:, 0]
        fav_state_daily = np.argmax(state_means_daily)
        
        if self.strategy == 'long-only':
            hmm_daily_state = ['Long' if s == fav_state_daily else 'Flat' for s in hidden_states_daily]
            kf_daily_state = ['Long' if s > 0 else 'Flat' for s in kf_slope_daily]
        else:  # long-short
            hmm_daily_state = ['Long' if s == fav_state_daily else 
                              'Short' if s == np.argmin(state_means_daily) else 
                              'Flat' for s in hidden_states_daily]
            kf_daily_state = ['Long' if s > 0 else 'Short' if s < 0 else 'Flat' for s in kf_slope_daily]

        logging.info(f"Daily predictions: {len(hmm_daily_state)} states")

        # Weekly predictions
        hidden_states_weekly = hmm_weekly.predict(f_test_weekly)
        state_means_weekly = hmm_weekly.means_[:, 0]
        fav_state_weekly = np.argmax(state_means_weekly)
        
        if self.strategy == 'long-only':
            hmm_weekly_state = ['Long' if s == fav_state_weekly else 'Flat' for s in hidden_states_weekly]
            kf_weekly_state = ['Long' if s > 0 else 'Flat' for s in kf_slope_weekly]
        else:  # long-short
            hmm_weekly_state = ['Long' if s == fav_state_weekly else 
                               'Short' if s == np.argmin(state_means_weekly) else 
                               'Flat' for s in hidden_states_weekly]
            kf_weekly_state = ['Long' if s > 0 else 'Short' if s < 0 else 'Flat' for s in kf_slope_weekly]

        weekly_df_raw = pd.DataFrame({
            'hmm_weekly': hmm_weekly_state, 
            'kf_weekly': kf_weekly_state
        }, index=f_test_weekly.index)
        weekly_df = weekly_df_raw.reindex(test_daily.index[1:], method='ffill')
        
        logging.info(f"Weekly predictions: {len(hmm_weekly_state)} states, reindexed to {len(weekly_df)} rows")

        if len(hmm_daily_state) != len(weekly_df):
            logging.warning(f"Length mismatch: daily={len(hmm_daily_state)}, weekly={len(weekly_df)}. Using minimum length.")

        ens_state = []
        min_len = min(len(hmm_daily_state), len(weekly_df))
        
        for i in range(min_len):
            daily_long = hmm_daily_state[i] == 'Long' and kf_daily_state[i] == 'Long'
            daily_short = (self.strategy == 'long-short' and 
                         hmm_daily_state[i] == 'Short' and kf_daily_state[i] == 'Short')
            
            weekly_long = weekly_df['hmm_weekly'].iloc[i] == 'Long' and weekly_df['kf_weekly'].iloc[i] == 'Long'
            weekly_short = (self.strategy == 'long-short' and 
                          weekly_df['hmm_weekly'].iloc[i] == 'Short' and weekly_df['kf_weekly'].iloc[i] == 'Short')
            
            if daily_long and weekly_long:
                ens_state.append('Long')
            elif daily_short and weekly_short:
                ens_state.append('Short')
            else:
                ens_state.append('Flat')
                
        return pd.Series(ens_state, index=test_daily.index[1:min_len+1])

    def ensemble_predict(self, hmm, test, kf_slope, features):
        try:
            models_daily, models_weekly = self.train_models_multi_res(self.train_data, test, features)
            states = self.voting_machine(*models_daily, *models_weekly, test)
            logging.info("Ensemble prediction completed")
            return states
        except Exception as e:
            logging.error(f"Ensemble prediction failed: {str(e)}")
            raise

    def simulate_trading(self, test, states):
        try:
            # Strategy returns
            shifted_states = states.shift(1).fillna('Flat')
            position_multiplier = (shifted_states == 'Long').astype(int) - (shifted_states == 'Short').astype(int)
            returns = test['Open'].pct_change() * position_multiplier
            cum_returns = (1 + returns).cumprod()
            
            # Buy and hold returns
            bnh_returns = test['Close'].pct_change().fillna(0.0)
            bnh_cum_returns = (1 + bnh_returns).cumprod()
            
            result = pd.DataFrame({
                'Close': test['Close'], 
                'ens_state': states,
                'returns': returns, 
                'cum_returns': cum_returns,
                'bnh_returns': bnh_returns,
                'bnh_cum_returns': bnh_cum_returns
            })
            logging.info("Trading simulation completed")
            return result
        except Exception as e:
            logging.error(f"Trading simulation failed: {str(e)}")
            raise

    def evaluate(self, results):
        try:
            # Strategy evaluation
            logging.info(f"ens_state sample: {results['ens_state'].head().tolist()}")
            cum_returns = results['cum_returns'].fillna(1.0)
            logging.info(f"Final cum_returns: {cum_returns.iloc[-1]}, length: {len(results)}")
            returns = results['returns'].fillna(0.0)
            
            # Calculate strategy metrics
            ann_ret = (cum_returns.iloc[-1] ** (365/len(results))) - 1
            sharpe = (returns.mean() / returns.std()) * np.sqrt(365)
            # Calculate Sortino ratio (using negative returns only for denominator)
            neg_returns = returns[returns < 0]
            sortino = (returns.mean() / neg_returns.std()) * np.sqrt(365) if len(neg_returns) > 0 else np.inf
            ann_vol = returns.std() * np.sqrt(365)
            drawdowns = cum_returns / cum_returns.cummax() - 1
            max_dd = drawdowns.min()
            
            # Count strategy switches
            ens_state = results['ens_state']
            switches = sum(ens_state.iloc[i] != ens_state.iloc[i-1] for i in range(1, len(ens_state)))
            
            # Buy and hold metrics
            bnh_returns = results['bnh_returns']
            bnh_cum_returns = results['bnh_cum_returns']
            bnh_ann_ret = (bnh_cum_returns.iloc[-1] ** (365/len(results))) - 1
            bnh_sharpe = (bnh_returns.mean() / bnh_returns.std()) * np.sqrt(365)
            # Calculate benchmark Sortino
            bnh_neg_returns = bnh_returns[bnh_returns < 0]
            bnh_sortino = (bnh_returns.mean() / bnh_neg_returns.std()) * np.sqrt(365) if len(bnh_neg_returns) > 0 else np.inf
            bnh_ann_vol = bnh_returns.std() * np.sqrt(365)
            bnh_drawdowns = bnh_cum_returns / bnh_cum_returns.cummax() - 1
            bnh_max_dd = bnh_drawdowns.min()
            
            # Create metrics table
            metrics_df = pd.DataFrame({
                'Metric': ['Annualized Return', 'Sharpe Ratio', 'Sortino Ratio', 'Annualized Volatility', 
                          'Maximum Drawdown', 'Final Cum Return', 'Switches'],
                'Strategy': [ann_ret, sharpe, sortino, ann_vol, max_dd, cum_returns.iloc[-1], switches],
                'Buy & Hold': [bnh_ann_ret, bnh_sharpe, bnh_sortino, bnh_ann_vol, bnh_max_dd, 
                              bnh_cum_returns.iloc[-1], 'N/A']
            })
            metrics_df.set_index('Metric', inplace=True)
            
            logging.info(f"Evaluation metrics:\n{metrics_df}")
            return metrics_df
        except Exception as e:
            logging.error(f"Evaluation failed: {str(e)}")
            raise



In [59]:
model = RegimeSwitchModel('bitcoin', strategy='long-short', train_pct=0.75)
data = model.load_data()
train, embargo, test = model.split_data(data)
train

2025-02-25 21:10:50,823 - INFO - Loaded 2005 rows for bitcoin
2025-02-25 21:10:50,824 - INFO - Split data: train=1503, embargo=20, test=482


Loading move data...
Loading treasury5YInflationExpectation data...
Loading creditSpreads data...
Loading vix data...


,Open,High,Low,Close,Volume,market_cap,log_close,move,treasury5YInflationExpectation,creditSpreads,vix,log_return,lnrange,rsi
Date,,,,,,,,,,,,,,
2017-01-17,828.31,828.31,828.31,828.31,1.147970e+09,1.407968e+10,6.719387,73.940002,1.84,4.06,11.87,0.001156,0.000000,32.692622
2017-01-18,904.15,904.15,904.15,904.15,8.758585e+08,1.445143e+10,6.806995,73.940002,1.86,4.01,12.48,0.013038,0.000000,39.248421
2017-01-19,873.93,873.93,873.93,873.93,6.137963e+08,1.436600e+10,6.773000,78.320000,1.90,3.99,12.78,-0.004994,0.000000,26.467355
2017-01-20,896.89,896.89,896.89,896.89,6.834912e+08,1.482282e+10,6.798933,76.709999,1.92,4.02,11.54,0.003829,0.000000,39.061261
2017-01-23,918.78,918.78,918.78,918.78,3.786662e+08,1.428176e+10,6.823047,75.519997,1.86,4.10,11.77,-0.000143,0.000000,51.516269
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-02-03,23757.00,24196.00,23520.00,23540.00,4.126704e+10,4.523126e+11,10.066456,98.989998,2.27,3.95,18.33,-0.000777,0.028336,70.510681
2023-02-06,23340.00,23411.00,22838.00,22946.00,3.396365e+10,4.387264e+11,10.040899,105.629997,2.33,3.99,19.43,-0.001693,0.024780,52.148879
2023-02-07,22948.00,23138.00,22692.00,22786.00,3.885059e+10,4.493912e+11,10.033902,103.150002,2.40,3.99,18.66,-0.000697,0.019464,47.916230


In [60]:
features = ['rsi']
states = model.ensemble_predict(None, test, None, features)
print("State Distribution:")
print(states.value_counts())

2025-02-25 21:10:51,244 - INFO - Kalman Filter trained successfully
2025-02-25 21:10:51,340 - INFO - Kalman Filter trained successfully
2025-02-25 21:10:51,344 - INFO - Multi-resolution models trained successfully
2025-02-25 21:10:51,348 - INFO - Daily predictions: 482 states
2025-02-25 21:10:51,352 - INFO - Weekly predictions: 102 states, reindexed to 481 rows
2025-02-25 21:10:51,354 - WARNING - Length mismatch: daily=482, weekly=481. Using minimum length.
2025-02-25 21:10:51,371 - INFO - Ensemble prediction completed


State Distribution:
Flat     367
Long      67
Short     47
Name: count, dtype: int64


In [61]:
results = model.simulate_trading(test, states)
metrics = model.evaluate(results)
print(results)


2025-02-25 21:10:51,514 - INFO - Trading simulation completed
2025-02-25 21:10:51,515 - INFO - ens_state sample: [nan, 'Flat', 'Flat', 'Flat', 'Flat']
2025-02-25 21:10:51,516 - INFO - Final cum_returns: 0.9497082132037938, length: 482
2025-02-25 21:10:51,528 - INFO - Evaluation metrics:
                         Strategy Buy & Hold
Metric                                      
Annualized Return       -0.038321   2.055064
Sharpe Ratio            -0.025166   2.199864
Sortino Ratio           -0.019719    3.79572
Annualized Volatility    0.256372   0.585136
Maximum Drawdown        -0.223339  -0.262319
Final Cum Return         0.949708   4.370112
Switches               147.000000        N/A


              Close ens_state   returns  cum_returns  bnh_returns  \
Date                                                                
2023-03-13  22096.0       NaN       NaN          NaN     0.000000   
2023-03-14  24179.0      Flat  0.000000     1.000000     0.094270   
2023-03-15  24759.0      Flat  0.000000     1.000000     0.023988   
2023-03-16  24471.0      Flat  0.000000     1.000000    -0.011632   
2023-03-17  25161.0      Flat -0.000000     1.000000     0.028197   
...             ...       ...       ...          ...          ...   
2025-02-10  96549.0      Flat -0.000000     0.981068    -0.000890   
2025-02-11  97400.0     Short  0.000000     0.981068     0.008814   
2025-02-12  95740.0      Flat -0.009791     0.971462    -0.017043   
2025-02-13  97836.0     Short -0.000000     0.971462     0.021893   
2025-02-14  96562.0      Flat -0.022393     0.949708    -0.013022   

            bnh_cum_returns  
Date                         
2023-03-13         1.000000  
2023-03-14  

In [62]:
import plotly.graph_objects as go

from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Create figure with secondary y-axis
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    vertical_spacing=0.1,
                    subplot_titles=(f"{model.symbol} Strategy vs Buy & Hold Returns", 
                                  "Market States"))

# Add returns traces on first subplot
fig.add_trace(go.Scatter(x=results.index, y=results['cum_returns'], 
                        mode='lines', name='Strategy Returns'),
              row=1, col=1)
fig.add_trace(go.Scatter(x=results.index, y=results['bnh_cum_returns'], 
                        mode='lines', name='Buy & Hold Returns'),
              row=1, col=1)

# Add states trace on second subplot
fig.add_trace(go.Scatter(x=states.index, y=states.astype('category').cat.codes,
                        mode='lines', name='Market State',
                        hovertext=states),
              row=2, col=1)

# Update layout
fig.update_layout(height=800,
                 showlegend=True)
fig.update_yaxes(title_text="Cumulative Returns", row=1, col=1)
fig.update_yaxes(title_text="State", row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1)

fig.show()

# Feature Exploration

In [50]:
model = RegimeSwitchModel('bitcoin')
data = model.load_data()
data

2025-02-25 21:04:58,240 - INFO - Loaded 2005 rows for bitcoin


Loading move data...
Loading treasury5YInflationExpectation data...
Loading creditSpreads data...
Loading vix data...


,Open,High,Low,Close,Volume,market_cap,log_close,move,treasury5YInflationExpectation,creditSpreads,vix,log_return,lnrange,rsi
Date,,,,,,,,,,,,,,
2017-01-17,828.31,828.31,828.31,828.31,1.147970e+09,1.407968e+10,6.719387,73.940002,1.84,4.06,11.87,0.001156,0.000000,32.692622
2017-01-18,904.15,904.15,904.15,904.15,8.758585e+08,1.445143e+10,6.806995,73.940002,1.86,4.01,12.48,0.013038,0.000000,39.248421
2017-01-19,873.93,873.93,873.93,873.93,6.137963e+08,1.436600e+10,6.773000,78.320000,1.90,3.99,12.78,-0.004994,0.000000,26.467355
2017-01-20,896.89,896.89,896.89,896.89,6.834912e+08,1.482282e+10,6.798933,76.709999,1.92,4.02,11.54,0.003829,0.000000,39.061261
2017-01-23,918.78,918.78,918.78,918.78,3.786662e+08,1.428176e+10,6.823047,75.519997,1.86,4.10,11.77,-0.000143,0.000000,51.516269
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-10,96454.00,97317.00,94761.00,96549.00,3.705499e+10,1.930743e+12,11.477806,87.682899,2.60,2.66,15.81,-0.000008,0.026616,36.643785
2025-02-11,96520.00,98334.00,95524.00,97400.00,3.645458e+10,1.898789e+12,11.486581,85.580002,2.64,2.66,16.02,0.000765,0.028992,40.435101
2025-02-12,97465.00,98467.00,94899.00,95740.00,4.711562e+10,1.936842e+12,11.469391,86.580002,2.66,2.65,15.89,-0.001497,0.036908,29.757979


In [51]:
import pandas as pd
import numpy as np
import ta
import plotly.express as px  # Added for interactive heatmap
from typing import Dict
import warnings
warnings.filterwarnings('ignore')

def explore_features(data: pd.DataFrame, verbose: bool = True) -> Dict[str, pd.DataFrame]:
    """
    Feature exploration pipeline outputting two sorted correlation lists with log_return
    and an interactive correlation heatmap.

    Args:
        data: Input DataFrame with OHLCV and optional macro features
        verbose: Whether to print analysis progress and display visualizations

    Returns:
        Dictionary containing processed data and analysis results
    """
    # Create a copy to avoid modifying the original data
    df = data.copy()
    
    if verbose:
        print("\nStarting feature exploration pipeline...")
    
    # Ensure required columns exist
    required_columns = ['Open', 'High', 'Low', 'Close', 'Volume']
    if not all(col in df.columns for col in required_columns):
        raise ValueError(f"DataFrame must contain all required columns: {required_columns}")

    # Calculate Technical Indicators
    if verbose:
        print("Calculating Technical Indicators...")
    
    # Momentum Indicators
    df['rsi_14'] = ta.momentum.RSIIndicator(close=df['Close'], window=14).rsi()
    df['rsi_7'] = ta.momentum.RSIIndicator(close=df['Close'], window=7).rsi()
    stoch = ta.momentum.StochasticOscillator(high=df['High'], low=df['Low'], close=df['Close'])
    df['stoch_k'] = stoch.stoch()
    df['stoch_d'] = stoch.stoch_signal()
    macd = ta.trend.MACD(close=df['Close'])
    df['macd'] = macd.macd()
    df['macd_signal'] = macd.macd_signal()
    df['macd_diff'] = macd.macd_diff()

    # Trend Indicators
    df['sma_20'] = ta.trend.SMAIndicator(close=df['Close'], window=20).sma_indicator()
    df['sma_50'] = ta.trend.SMAIndicator(close=df['Close'], window=50).sma_indicator()
    df['ema_20'] = ta.trend.EMAIndicator(close=df['Close'], window=20).ema_indicator()
    df['adx'] = ta.trend.ADXIndicator(high=df['High'], low=df['Low'], close=df['Close']).adx()

    # Volatility Indicators
    bb = ta.volatility.BollingerBands(close=df['Close'])
    df['bb_upper'] = bb.bollinger_hband()
    df['bb_middle'] = bb.bollinger_mavg()
    df['bb_lower'] = bb.bollinger_lband()
    df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['bb_middle']
    df['atr'] = ta.volatility.AverageTrueRange(high=df['High'], low=df['Low'], 
                                              close=df['Close']).average_true_range()

    # Volume Indicators
    df['volume_ma_20'] = df['Volume'].rolling(window=20).mean()
    df['volume_ratio'] = df['Volume'] / df['volume_ma_20']
    df['obv'] = ta.volume.OnBalanceVolumeIndicator(close=df['Close'], 
                                                  volume=df['Volume']).on_balance_volume()

    # Custom Features
    if verbose:
        print("Calculating Custom Features...")
    
    df['log_return'] = np.log(df['Close']).diff()
    df['realized_vol_20'] = df['log_return'].rolling(window=20).std() * np.sqrt(252)
    df['high_low_range'] = (df['High'] - df['Low']) / df['Close']
    df['price_sma20_ratio'] = df['Close'] / df['sma_20']
    df['price_sma50_ratio'] = df['Close'] / df['sma_50']

    # Macro Features (if present)
    if verbose:
        print("Calculating Macro Feature Transformations...")
    
    macro_features = ['vix', 'creditSpreads', 'move', 'treasury5YInflationExpectation']
    for feature in macro_features:
        if feature in df.columns:
            df[f'{feature}_ret'] = df[feature].pct_change()
            df[f'{feature}_ma7'] = df[feature].rolling(window=7).mean()
            df[f'{feature}_week_ratio'] = df[feature] / df[f'{feature}_ma7']

    # Clean data
    df_clean = df.replace([np.inf, -np.inf], np.nan).dropna()

    # Feature Analysis
    if verbose:
        print("\nPerforming Feature Analysis...")
    
    # Select features for correlation analysis
    feature_patterns = ['_ret', 'rsi_', 'stoch_', 'macd', 'sma_', 'ema_', 'bb_', 
                        'volume_', 'realized_vol_', 'price_', '_ratio', 'adx', 'atr', 'obv']
    matching_columns = [col for col in df_clean.columns if any(pattern in col for pattern in feature_patterns)]
    features_to_analyze = list(dict.fromkeys(['log_return'] + matching_columns))

    # Calculate correlation matrix
    correlation_matrix = df_clean[features_to_analyze].corr()

    # Extract and sort correlations with log_return
    if 'log_return' in correlation_matrix.columns:
        correlations_with_returns = correlation_matrix['log_return']
        
        # Separate positive and negative correlations
        positive_corrs = correlations_with_returns[correlations_with_returns > 0].sort_values(ascending=False)
        negative_corrs = correlations_with_returns[correlations_with_returns < 0].sort_values(ascending=True)
        
        # Output the sorted lists
        if verbose:
            print("\n**Positive Correlations with log_return (High to Low):**")
            print(positive_corrs.to_string())
            print("\n**Negative Correlations with log_return (Low to High):**")
            print(negative_corrs.to_string())
    else:
        print("Warning: 'log_return' not found in correlation matrix.")

    # Generate interactive heatmap if verbose
    if not correlation_matrix.empty and verbose:
        print("\nGenerating interactive correlation heatmap...")
        fig = px.imshow(correlation_matrix, 
                        color_continuous_scale='RdBu', 
                        zmin=-1, 
                        zmax=1,
                        title='Feature Correlation Heatmap')
        fig.update_layout(width=1000, height=1000)
        fig.update_xaxes(tickangle=45)
        fig.show()

    # Prepare results
    results = {
        'processed_data': df_clean,
        'correlation_matrix': correlation_matrix,
        'positive_correlations': positive_corrs if 'log_return' in correlation_matrix.columns else pd.Series(dtype=float),
        'negative_correlations': negative_corrs if 'log_return' in correlation_matrix.columns else pd.Series(dtype=float)
    }

    if verbose:
        print("\nFeature exploration completed!")
    
    return results

In [52]:
# Run the function
results = explore_features(data)

# Access results
processed_data = results['processed_data']


Starting feature exploration pipeline...
Calculating Technical Indicators...
Calculating Custom Features...
Calculating Macro Feature Transformations...

Performing Feature Analysis...

**Positive Correlations with log_return (High to Low):**
log_return                                   1.000000
rsi_7                                        0.458895
stoch_k                                      0.426899
price_sma20_ratio                            0.391567
rsi_14                                       0.357440
price_sma50_ratio                            0.259094
stoch_d                                      0.198498
macd_diff                                    0.146726
volume_ratio                                 0.114351
treasury5YInflationExpectation_ret           0.107217
treasury5YInflationExpectation_week_ratio    0.079849
adx                                          0.068325
macd                                         0.065640
bb_width                                     0.048910



Feature exploration completed!
